In [6]:
from pathlib import Path

import numpy as np
from astropy.table import Table, vstack, unique
from tqdm.auto import tqdm


def merge_ecsv_files(
    input_dir,
    output_file,
    duplicate_column="source_id"
):


    input_dir = Path(input_dir)
    output_file = Path(output_file)

    files = sorted(input_dir.rglob("*.ecsv"))

    print(f"Archivos encontrados: {len(files)}")

    if not files:
        raise FileNotFoundError(
            f"No hay ficheros .ecsv en {input_dir}"
        )

    tables = []

    for file in tqdm(files, desc="Leyendo ECSV"):
        try:

            table = Table.read(file, format="ascii.ecsv")
            if len(table) > 0:
                tables.append(table)

        except Exception as exc:

            print( f"\nError leyendo {file}: {exc}" )


    if not tables:
        raise ValueError(
            f"Los {len(files)} ficheros de {input_dir} están vacíos "
            f"o no se han podido leer"
        )

    combined = vstack( tables, metadata_conflicts="silent" )

    print( f"Filas antes de eliminar duplicados: "f"{len(combined)}" )

    combined = unique(
        combined,
        keys=duplicate_column,
        keep="first"
    )

    print( f"Filas después de eliminar duplicados: "f"{len(combined)}")

  
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    combined.write(
        output_file,
        format="ascii.ecsv",
        overwrite=True
    )

    print(f"\nArchivo guardado en:\n{output_file}")

    return combined

In [7]:
# Localizamos la carpeta 'solution' subiendo desde el directorio de trabajo, sin
# suponer dónde arranca el kernel. Su marca es el fichero 'pyproject.toml'.
SOLUTION_DIR = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "solution")
    if (candidate / "pyproject.toml").is_file()
)

# Tablas de Gaia del cruce con SDSS, una por bloque de descarga
input_dir = SOLUTION_DIR / "dataset_extraction_pipeline" / "data" / "02_gaia_crossmatch"


In [8]:
output_file = SOLUTION_DIR / "data" / "gaia_data.ecsv"


In [9]:
gaia_hr_table = merge_ecsv_files(
    input_dir=input_dir,
    output_file=output_file,
    duplicate_column="source_id"
)

Archivos encontrados: 17496


Leyendo ECSV:   0%|          | 0/17496 [00:00<?, ?it/s]

Filas antes de eliminar duplicados: 370543
Filas después de eliminar duplicados: 121890

Archivo guardado en:
/home/javier-cacho/repos/master-ia/trabajo-fin-master/solution/data/gaia_data.ecsv
